In [ ]:
import requests

url = "https://rickandmortyapi.com/api/character/?page=1"
response = requests.get(url)


In [ ]:
# Transformer la réponse en objet JSON
import json
data = response.json()["results"]
print(data)


In [ ]:
def json_to_text(obj, prefix = ""):
        lines = []
        if isinstance(obj, dict):
            for k, v in obj.items():
                lines.append(json_to_text(v, prefix=f"{prefix}{k}."))
        elif isinstance(obj, list):
            for i, item in enumerate(obj):
                lines.append(json_to_text(item, prefix=f"{prefix}{i}."))
        else:
                lines.append(f"{prefix[:-1]} : {obj}")
        return "\n".join(lines)


In [ ]:
from langchain_core.documents import Document
docs_api = [
    Document(
        page_content=json_to_text(character), 
        metadata={"character_id": character.get("id", None)}
    ) for character in data
]

In [ ]:
print(f"Nombre de documents : {len(docs_api)}")
print("="*50)
print (f"Exemple de document : {docs_api[0].page_content}")
print("="*50)
print (f"Exemple de metadata : {docs_api[0].metadata}")

In [ ]:
# chunker les documents nettoyés
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter()

chunks_api = splitter.split_documents(docs_api)
print(f"Nombre de chunks : {len(chunks_api)}")
#print(chunks_api[0].page_content)

# BDD

In [ ]:
import sqlite3

conn = sqlite3.connect("support.db")

print("Connexion réussie !")

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS tickets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        client TEXT,
        priority TEXT,
        description TEXT,
        created_at TEXT
    )
""")

conn.commit()

In [ ]:
insert_query = """
    INSERT INTO tickets (client, priority, description, created_at)
    VALUES (?, ?, ?, ?)
"""
insert_data = [
    ("Client A", "High", "Problème critique", "2024-06-01 10:00:00"),
    ("Client B", "Medium", "Problème mineur", "2024-06-02 11:30:00"),
    ("Client C", "Low", "Demande d'information", "2024-06-03 14:15:00"),
]

cursor.executemany(insert_query, insert_data)
conn.commit()

In [ ]:
#voir ce qu'il y a dans la table
cursor.execute("SELECT * FROM tickets")
rows = cursor.fetchall()
for row in rows:
    print(row) 


In [ ]:
import pandas as pd

df = pd.read_sql_query("SELECT * FROM tickets", conn)
print(df)

In [ ]:
#convertir le dataframe en text
def sql_row_to_text(df):
    return "\n".join([f" id: {row['id']} \n Client: {row['client']} \n Priority: {row['priority']} \n Description: {row['description']} \n Created At: {row['created_at']}" for index, row in df.iterrows()])

In [ ]:
df_text = sql_row_to_text(df.iloc[:5])  # Convertir les 5 premières lignes du DataFrame en texte
print(df_text)

In [ ]:
# Transformer chaque ligne SQL en Document
from langchain_core.documents import Document

docs_sql = []
for _, row in df.iterrows():
    text = sql_row_to_text(row.to_frame().T)
    doc = Document(
        page_content=text,
        metadata={
            "source": "sql",
            "table": "tickets",
            "id": row["id"],
            "priority": row["priority"],
            "created_at": row["created_at"],
        },
    )
    docs_sql.append(doc)

print("nb_docs", len(docs_sql))
if docs_sql:
    print("page_content", docs_sql[0].page_content)
    print("metadata", docs_sql[0].metadata)


In [ ]:
#chunker les documents SQL
chunks_sql = splitter.split_documents(docs_sql)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # taille idéale d’un chunk en caractères
    chunk_overlap=150,    # chevauchement entre les chunks
    separators=["\n\n", "\n", ".", " ", ""]  # ordre logique de découpe
)

chunks_sql = splitter.split_documents(docs_sql)
print(f"Nombre de chunks SQL : {len(chunks_sql)}")
print(f"Exemple de chunk SQL : {chunks_sql[0].page_content}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pandas as pd

class Ingest_SQL:
    def __init__(self, conn, chunk_size=800, chunk_overlap=100):
        self.conn = conn
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", " ", ""],
        )

    def row_to_text(self, row):
        return (
            f"id : {row['id']}\n"
            f"client : {row['client']}\n"
            f"priority : {row['priority']}\n"
            f"description : {row['description']}\n"
            f"created_at : {row['created_at']}"
        )

    def sql_to_documents(self, query):
        df = pd.read_sql_query(query, self.conn)
        docs = []
        for _, row in df.iterrows():
            text = self.row_to_text(row)
            meta = row.to_dict()
            meta["source"] = "sql"
            meta["table"] = "tickets"
            docs.append(Document(page_content=text, metadata=meta))
        return docs

    def process(self, query):
        docs = self.sql_to_documents(query)
        return self.splitter.split_documents(docs)


In [ ]:
#appliquer la classe Ingest_SQL
ingest_sql = Ingest_SQL(conn)
query = "SELECT * FROM tickets"
chunks_from_sql = ingest_sql.process(query)
print(f"Nombre de chunks à partir de SQL : {len(chunks_from_sql)}")